In [4]:
# === Merge FACE -> SPEAKER (1 celda) =========================================
from pathlib import Path
import re, csv, bisect

def _pick_single_subs_from_srts_dual() -> Path:
    d = Path("srts_dual")
    if not d.exists():
        raise FileNotFoundError("No existe la carpeta 'srts_dual'.")
    cands = list(d.glob("*.srt"))
    if not cands:
        raise FileNotFoundError("No se encontró ningún .srt en 'srts_dual'.")
    if len(cands) > 1:
        raise RuntimeError(f"Se esperaba un único .srt en 'srts_dual', encontrados: {[c.name for c in cands]}")
    return cands[0]

try:
    SUBS_IN = Path(SUBS_IN)
except NameError:
    SUBS_IN = _pick_single_subs_from_srts_dual()

try:
    FACES_IN = Path(FACES_IN)
except NameError:
    FACES_IN = Path("diarization_elements") / "talking_faces.srt"

# --- Resolver PROJECT_ROOT de forma robusta a partir de SUBS_IN ---
def resolve_project_root(subs_path: Path) -> Path:
    p = subs_path.resolve()
    # Caso 1: si en la cadena de padres existe una carpeta 'code', usamos su padre como root
    for parent in p.parents:
        if parent.name.lower() == "code":
            return parent.parent
    # Caso 2: si el srt está bajo 'srts_dual', entonces root es el padre de 'code' (p.parent.parent)
    if p.parent.name.lower() == "srts_dual" and len(p.parents) >= 2:
        return p.parents[1]  # p.parent.parent
    # Caso 3: si el SRT está en la misma carpeta del notebook o cualquier otra, guardamos en su abuelo si existe
    return p.parents[1] if len(p.parents) > 1 else p.parent

PROJECT_ROOT = resolve_project_root(SUBS_IN)
OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Nombre de salida: igual al original + "_diarizado.srt" dentro de ../output
OUT_SRT = OUT_DIR / f"{SUBS_IN.stem}_diarizado.srt"

OUT_MAP = None
LABEL_POSITION = "prefix"   # "prefix": SPEAKER_n: texto | "suffix": texto (SPEAKER_n)

# ----------------- Utilidades -----------------
TIME_RE = re.compile(r"(\d{2}:\d{2}:\d{2},\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2},\d{3})")
FACE_LINE_RE = re.compile(r"(?:^|\s)(FACE[_ ]\d+)\b", flags=re.IGNORECASE)
SPEAKER_PREFIX_RE = re.compile(r"^(SPEAKER[_ ]\d+)(:?\s*)", flags=re.IGNORECASE)
FACE_PREFIX_RE    = re.compile(r"^(FACE[_ ]\d+)(:?\s*)",    flags=re.IGNORECASE)

def parse_time_srt(t: str) -> float:
    hh, mm, rest = t.split(":")
    ss, ms = rest.split(",")
    return int(hh)*3600 + int(mm)*60 + int(ss) + int(ms)/1000.0

def fmt_time_srt(s: float) -> str:
    ms_total = int(round(s * 1000))
    hh, rem = divmod(ms_total, 3600_000)
    mm, rem = divmod(rem, 60_000)
    ss, ms = divmod(rem, 1000)
    return f"{hh:02d}:{mm:02d}:{ss:02d},{ms:03d}"

def parse_srt_blocks(path: Path):
    text = path.read_text(encoding="utf-8", errors="replace")
    blocks = re.split(r"\r?\n\s*\r?\n", text.strip())
    cues = []
    for b in blocks:
        lines = [ln.rstrip() for ln in b.splitlines() if ln.strip() != ""]
        if not lines:
            continue
        t_idx = None
        for i, ln in enumerate(lines):
            if "-->" in ln:
                t_idx = i; break
        if t_idx is None:
            continue
        m = TIME_RE.search(lines[t_idx])
        if not m:
            continue
        start = parse_time_srt(m.group(1))
        end   = parse_time_srt(m.group(2))
        text_lines = lines[t_idx+1:] if t_idx+1 < len(lines) else []
        cues.append((start, end, text_lines))
    return cues

def overlap(a, b):
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))

def load_faces(faces_srt: Path):
    face_cues = parse_srt_blocks(faces_srt)
    faces = []
    for s, e, txt_lines in face_cues:
        if not txt_lines:
            continue
        m = FACE_LINE_RE.search(txt_lines[0].strip())
        if not m:
            continue
        label = m.group(1).upper().replace(" ", "_")
        faces.append((s, e, label))
    faces.sort(key=lambda x: x[0])
    return faces

def assign_face_to_subs(subs, faces):
    if not faces:
        return [(s,e,lines,None) for (s,e,lines) in subs]

    faces_by_start = sorted(faces, key=lambda x: x[0])
    faces_by_end   = sorted(faces, key=lambda x: x[1])
    face_starts    = [fs for fs,_,_ in faces_by_start]
    face_ends      = [fe for _,fe,_ in faces_by_end]

    assigned = []
    for s, e, lines in subs:
        best_label, best_ov = None, 0.0
        for fs, fe, flab in faces:
            ov = overlap((s,e), (fs,fe))
            if ov > best_ov:
                best_ov, best_label = ov, flab
        if best_ov == 0.0 or best_label is None:
            idx_prev = bisect.bisect_right(face_ends, s) - 1
            if idx_prev >= 0:
                best_label = faces_by_end[idx_prev][2]
            else:
                idx_next = bisect.bisect_left(face_starts, e)
                best_label = faces_by_start[idx_next][2] if idx_next < len(faces_by_start) else None
        assigned.append((s, e, lines, best_label))
    return assigned

def apply_speaker_labels(assigned, label_position="prefix", mapping_csv: Path | None = None):
    face2spk, next_id = {}, 1
    out_blocks = []
    for s, e, lines, face_label in assigned:
        new_lines = list(lines) if lines else []
        if new_lines:
            first = new_lines[0].strip()
            first = SPEAKER_PREFIX_RE.sub("", first)
            first = FACE_PREFIX_RE.sub("", first)
            first = first.lstrip(": ").lstrip()
            new_lines[0] = first
        if face_label:
            if face_label not in face2spk:
                face2spk[face_label] = f"SPEAKER_{next_id}"; next_id += 1
            spk = face2spk[face_label]
            if new_lines:
                new_lines[0] = (f"{spk}: {new_lines[0]}" if label_position == "prefix"
                                else f"{new_lines[0]} ({spk})")
            else:
                new_lines = [spk]
        out_blocks.append((s, e, new_lines))
    if mapping_csv is not None:
        with mapping_csv.open("w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(["FACE_label","SPEAKER_label"])
            for face, spk in face2spk.items():
                w.writerow([face, spk])
    return out_blocks

def write_srt(blocks, out_path: Path):
    with out_path.open("w", encoding="utf-8") as f:
        for i, (s, e, lines) in enumerate(blocks, 1):
            f.write(f"{i}\n")
            f.write(f"{fmt_time_srt(s)} --> {fmt_time_srt(e)}\n")
            if lines:
                for ln in lines:
                    f.write(f"{ln}\n")
            f.write("\n")

# ----------------- Ejecución -----------------
print(f"[INFO] Subtítulos base -> {SUBS_IN}")
print(f"[INFO] Caras -> {FACES_IN}")
print(f"[INFO] Guardando en  -> {OUT_SRT}")

subs  = parse_srt_blocks(SUBS_IN)
faces = load_faces(FACES_IN)
assigned = assign_face_to_subs(subs, faces)
blocks   = apply_speaker_labels(assigned, label_position=LABEL_POSITION, mapping_csv=OUT_MAP)
write_srt(blocks, OUT_SRT)

# --- Verificación y debug ---
print(f"[DBG] CWD          -> {Path.cwd()}")
print(f"[DBG] SUBS_IN      -> {SUBS_IN.resolve()}")
print(f"[DBG] PROJECT_ROOT -> {PROJECT_ROOT.resolve()}")
print(f"[DBG] OUT_DIR      -> {OUT_DIR.resolve()}")
print(f"[DBG] OUT_SRT      -> {OUT_SRT.resolve()}")

if OUT_SRT.exists():
    size = OUT_SRT.stat().st_size
    print(f"[OK] SRT final -> {OUT_SRT.resolve()} ({size} bytes)")
    if size == 0:
        print("[WARN] El archivo está vacío. Revisa si 'subs' tenía cues y 'faces' coincidencias.")
else:
    print("[ERROR] No se encontró el archivo de salida tras escribir.")

if OUT_MAP:
    print(f"[OK] Mapeo FACE->SPEAKER -> {OUT_MAP.resolve()}")
print(f"Cues: {len(blocks)} | Caras distintas: {len({f for *_,f in assigned if f})}")
# ============================================================================


[INFO] Subtítulos base -> srts_dual\sub_es_en.srt
[INFO] Caras -> diarization_elements\talking_faces.srt
[INFO] Guardando en  -> C:\Users\carlos.basallote\Desktop\TFM\TFM\output\sub_es_en_diarizado.srt
[DBG] CWD          -> c:\Users\carlos.basallote\Desktop\TFM\TFM\code
[DBG] SUBS_IN      -> C:\Users\carlos.basallote\Desktop\TFM\TFM\code\srts_dual\sub_es_en.srt
[DBG] PROJECT_ROOT -> C:\Users\carlos.basallote\Desktop\TFM\TFM
[DBG] OUT_DIR      -> C:\Users\carlos.basallote\Desktop\TFM\TFM\output
[DBG] OUT_SRT      -> C:\Users\carlos.basallote\Desktop\TFM\TFM\output\sub_es_en_diarizado.srt
[OK] SRT final -> C:\Users\carlos.basallote\Desktop\TFM\TFM\output\sub_es_en_diarizado.srt (45338 bytes)
Cues: 296 | Caras distintas: 21


In [5]:
print(f"[INFO] Salida -> {(OUT_DIR / f'{SUBS_IN.stem}_diarizado.srt').resolve()}")


[INFO] Salida -> C:\Users\carlos.basallote\Desktop\TFM\TFM\output\sub_es_en_diarizado.srt
